**Kaushik Budur**

MyMav ID: 1002224112

Code to compute the 3-way orthologs across all three species.


In [2]:

from google.colab import files
uploaded = files.upload()

import pandas as pd

def best_hits(filename: str):
    """Return dict {query: best_subject} using lowest e-value per query."""
    df = pd.read_csv(filename, sep="\t", header=None, engine="python")
    df = df.iloc[:, :3]
    df.columns = ["query", "subject", "evalue"]
    df["evalue"] = pd.to_numeric(df["evalue"], errors="coerce")
    df = df.sort_values(["query", "evalue"], ascending=[True, True])
    best = df.drop_duplicates(subset="query", keep="first")
    return dict(zip(best["query"], best["subject"]))

def rbh(file1, file2):
    """Find reciprocal best hits between two BLAST outputs."""
    d1 = best_hits(file1)
    d2 = best_hits(file2)
    return {(a, b) for a, b in d1.items() if d2.get(b) == a}

# STEP 2: Build RBH sets for each species pair
rbh_TL = rbh("Tcas_query_v_Ldec_subject.txt", "Ldec_query_v_Tcas_subject.txt")
rbh_AL = rbh("Agla_query_v_Ldec_subject.txt", "Ldec_query_v_Agla_subject.txt")
rbh_AT = rbh("Agla_query_v_Tcas_subject.txt", "Tcas_query_v_Agla_subject.txt")

# STEP 3: Convert to quick lookup dicts
T2L = {t:l for t,l in rbh_TL}
L2T = {l:t for t,l in rbh_TL}

A2L = {a:l for a,l in rbh_AL}
L2A = {l:a for a,l in rbh_AL}

A2T = {a:t for a,t in rbh_AT}
T2A = {t:a for a,t in rbh_AT}

# STEP 4: Find consistent 3-way ortholog groups
groups = []
for agla, tcas in A2T.items():
    l_from_agla = A2L.get(agla)
    l_from_tcas = T2L.get(tcas)
    if l_from_agla is None or l_from_tcas is None:
        continue
    if l_from_agla != l_from_tcas:
        continue
    ldec = l_from_agla
    if (L2A.get(ldec) == agla) and (L2T.get(ldec) == tcas) and (T2A.get(tcas) == agla):
        groups.append((agla, tcas, ldec))

groups = sorted(set(groups))
count_groups = len(groups)

# STEP 5: Print only the 3-way count
print(f"Number of 3-way orthologous groups (Agla–Tcas–Ldec): {count_groups}")


Saving Tcas_query_v_Ldec_subject.txt to Tcas_query_v_Ldec_subject.txt
Saving Ldec_query_v_Agla_subject.txt to Ldec_query_v_Agla_subject.txt
Saving Tcas_query_v_Agla_subject.txt to Tcas_query_v_Agla_subject.txt
Saving Ldec_query_v_Tcas_subject.txt to Ldec_query_v_Tcas_subject.txt
Saving Agla_query_v_Tcas_subject.txt to Agla_query_v_Tcas_subject.txt
Saving Agla_query_v_Ldec_subject.txt to Agla_query_v_Ldec_subject.txt
Number of 3-way orthologous groups (Agla–Tcas–Ldec): 7360


Number of 3-way orthologous groups are there between all three species?

7,360

In [3]:
def parse_blast_file(filename):
    """Parse BLAST tabular output to get best hit per query (first hit assumed best)."""
    best_hits = {}
    with open(filename) as f:
        for line in f:
            if line.strip() == '' or line.startswith('#'):
                continue
            parts = line.strip().split()
            query, subject = parts[0], parts[1]
            if query not in best_hits:
                best_hits[query] = subject
    return best_hits

# File pairs mapped to output filenames
file_pairs = [
    ("Agla_query_v_Tcas_subject.txt", "Tcas_query_v_Agla_subject.txt", "reciprocal_hits_Agla_Tcas.txt"),
    ("Ldec_query_v_Agla_subject.txt", "Agla_query_v_Ldec_subject.txt", "reciprocal_hits_Ldec_Agla.txt"),
    ("Tcas_query_v_Ldec_subject.txt", "Ldec_query_v_Tcas_subject.txt", "reciprocal_hits_Tcas_Ldec.txt"),
]

for file_a, file_b, out_filename in file_pairs:
    hits_a = parse_blast_file(file_a)
    hits_b = parse_blast_file(file_b)
    reciprocal_hits = []
    for query, hit in hits_a.items():
        if hit in hits_b and hits_b[hit] == query:
            reciprocal_hits.append((query, hit))

    with open(out_filename, 'w') as outfile:
        outfile.write(f'Reciprocal best hits for {file_a} and {file_b}:\n')
        outfile.write(f'Total pairs: {len(reciprocal_hits)}\n')
        for pair in reciprocal_hits:
            outfile.write(f'{pair[0]}\t{pair[1]}\n')

    print(f'Saved {len(reciprocal_hits)} reciprocal hits to {out_filename}')


Saved 10250 reciprocal hits to reciprocal_hits_Agla_Tcas.txt
Saved 9921 reciprocal hits to reciprocal_hits_Ldec_Agla.txt
Saved 9509 reciprocal hits to reciprocal_hits_Tcas_Ldec.txt


In [4]:
def read_pairs(filename):
    pairs = set()
    with open(filename) as f:
        for line in f:
            if not line.strip() or line.startswith('Reciprocal') or line.startswith('Total') or line.startswith('='):
                continue
            parts = line.strip().split()
            if len(parts) == 2:
                pairs.add(tuple(parts))
    return pairs

# Load 2-way RBBH pairs
agla_tcas = read_pairs('reciprocal_hits_Agla_Tcas.txt')
agla_ldec = read_pairs('reciprocal_hits_Ldec_Agla.txt')
ldec_tcas = read_pairs('reciprocal_hits_Tcas_Ldec.txt')

# Create lookup dictionaries for all three directions
agla2tcas = {a: t for a, t in agla_tcas}
tcas2ldec = {t: l for t, l in ldec_tcas}
ldec2agla = {l: a for l, a in agla_ldec}

# 1) Agla-Tcas-Ldec
trios_agla_tcas_ldec = []
for agla, tcas in agla2tcas.items():
    if tcas in tcas2ldec:
        ldec = tcas2ldec[tcas]
        if ldec in ldec2agla and ldec2agla[ldec] == agla:
            trios_agla_tcas_ldec.append((agla, tcas, ldec))

# 2) Tcas-Ldec-Agla
trios_tcas_ldec_agla = []
for tcas, ldec in tcas2ldec.items():
    if ldec in ldec2agla:
        agla = ldec2agla[ldec]
        if agla in agla2tcas and agla2tcas[agla] == tcas:
            trios_tcas_ldec_agla.append((tcas, ldec, agla))

# 3) Ldec-Agla-Tcas
trios_ldec_agla_tcas = []
for ldec, agla in ldec2agla.items():
    if agla in agla2tcas:
        tcas = agla2tcas[agla]
        if tcas in tcas2ldec and tcas2ldec[tcas] == ldec:
            trios_ldec_agla_tcas.append((ldec, agla, tcas))

# Save output to respective files
with open('3way_Agla_Tcas_Ldec.txt', 'w') as f1, \
     open('3way_Tcas_Ldec_Agla.txt', 'w') as f2, \
     open('3way_Ldec_Agla_Tcas.txt', 'w') as f3:
    f1.write('Agla\tTcas\tLdec\n')
    for trio in trios_agla_tcas_ldec:
        f1.write(f'{trio[0]}\t{trio[1]}\t{trio[2]}\n')
    f2.write('Tcas\tLdec\tAgla\n')
    for trio in trios_tcas_ldec_agla:
        f2.write(f'{trio[0]}\t{trio[1]}\t{trio[2]}\n')
    f3.write('Ldec\tAgla\tTcas\n')
    for trio in trios_ldec_agla_tcas:
        f3.write(f'{trio[0]}\t{trio[1]}\t{trio[2]}\n')

# Print counts
print('Number of 3-way matches per combination:')
print(f'Agla-Tcas-Ldec: {len(trios_agla_tcas_ldec)}')
print(f'Tcas-Ldec-Agla: {len(trios_tcas_ldec_agla)}')
print(f'Ldec-Agla-Tcas: {len(trios_ldec_agla_tcas)}')


Number of 3-way matches per combination:
Agla-Tcas-Ldec: 7360
Tcas-Ldec-Agla: 7360
Ldec-Agla-Tcas: 7360
